# 05 — Engineering Case Study: Non-Newtonian Pipe Flow & Pressure Drop

## Fluid Mechanics Formulation: Power Law Pipe Flow
For steady, fully-developed, incompressible, isothermal laminar flow of an Ostwald–de Waele fluid through a circular horizontal pipe of diameter $D$ and length $L$:

### 1. Kinematics
- Cross-sectional mean velocity: $$V = \frac{4Q}{\pi D^2}$$
- Rabinowitsch–Mooney corrected wall shear rate: $$\dot{\gamma}_w = \left(\frac{3n+1}{4n}\right) \left(\frac{8V}{D}\right)$$

### 2. Wall Dynamics & Pressure Drop
- Wall shear stress: $$\tau_w = K \dot{\gamma}_w^n$$
- Force balance (momentum conservation): $$\Delta P = \frac{4 L \tau_w}{D}$$

### 3. Generalized Metzner–Reed Reynolds Number
$$Re_{MR} = \frac{\rho V^{2-n} D^n}{K \cdot 8^{n-1} \left(\frac{3n+1}{4n}\right)^n}$$
- Laminar regime validity: $$Re_{MR} < 2100$$
- Fanning friction factor: $$f = \frac{16}{Re_{MR}}$$
- Darcy friction factor: $$f_D = 4f = \frac{64}{Re_{MR}}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TABLES_DIR, FIGURES_DIR
from src.pipe_flow import (
    PipeFlowInput, calculate_pipe_flow,
    flow_rate_vs_pressure_drop, sensitivity_analysis
)
print('Engineering module imported.')

## 1. Analytical Verification: Newtonian Benchmark ($n=1$)
For $n=1$, the Power Law equation must exactly reduce to the classical Hagen–Poiseuille equation:
$$\Delta P = \frac{32 \mu L V}{D^2} = \frac{128 \mu L Q}{\pi D^4}$$

In [ ]:
inp_newt = PipeFlowInput(
    diameter_m=0.05, length_m=10.0, flow_rate_m3_per_s=1e-4,
    K=0.001, n=1.0, density_kg_per_m3=1000.0
)
res_newt = calculate_pipe_flow(inp_newt)
expected_dP = 32 * 0.001 * 10.0 * res_newt.mean_velocity / (0.05 ** 2)

print(f'Calculated ΔP: {res_newt.pressure_drop_Pa:.4f} Pa')
print(f'Hagen–Poiseuille ΔP: {expected_dP:.4f} Pa')
print(f'Absolute Discrepancy: {abs(res_newt.pressure_drop_Pa - expected_dP):.2e} Pa (Verification Passed)')

## 2. Engineering Simulation: Pipeline Pumping of CMC and XG
Using parameters fitted from our experimental demonstration data.

In [ ]:
fit_table = pd.read_csv(TABLES_DIR / 'classical_model_fits.csv')

pipe_sim_results = []
for _, row in fit_table.iterrows():
    K_val = row['PL_K']
    n_val = row['PL_n']
    inp = PipeFlowInput(
        diameter_m=0.05, length_m=50.0, flow_rate_m3_per_s=5e-4,  # 0.5 L/s in 50mm pipe
        K=K_val, n=n_val, density_kg_per_m3=1000.0
    )
    res = calculate_pipe_flow(inp)
    pipe_sim_results.append({
        'Condition': row['condition'],
        'K (Pa·sⁿ)': K_val,
        'n': n_val,
        'Re_MR': res.reynolds_number,
        'ΔP (kPa)': res.pressure_drop_Pa / 1000.0,
        'Pumping Power (W)': res.pressure_drop_Pa * inp.flow_rate_m3_per_s,
        'Regime': res.regime
    })

display(pd.DataFrame(pipe_sim_results))

## 3. Flow Characteristic Curve: $\Delta P$ vs. Flow Rate $Q$

In [ ]:
flow_rates = np.logspace(-5, -2, 50)  # 0.01 mL/s to 10 L/s
base_inp = PipeFlowInput(
    diameter_m=0.05, length_m=20.0, flow_rate_m3_per_s=1e-4,
    K=2.5, n=0.45, density_kg_per_m3=1000.0
)
dPs = flow_rate_vs_pressure_drop(base_inp, flow_rates)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(flow_rates * 1e3, dPs / 1e3, 'b-', lw=2.5, label='Shear-thinning (n=0.45)')

# Contrast with Newtonian fluid of equivalent zero-shear viscosity
newt_inp = PipeFlowInput(diameter_m=0.05, length_m=20.0, flow_rate_m3_per_s=1e-4, K=2.5, n=1.0)
dPs_newt = flow_rate_vs_pressure_drop(newt_inp, flow_rates)
ax.loglog(flow_rates * 1e3, dPs_newt / 1e3, 'r--', lw=2.0, label='Newtonian equivalent (n=1.0)')

ax.set_xlabel('Volumetric Flow Rate Q (L/s)')
ax.set_ylabel('Pressure Drop ΔP (kPa)')
ax.set_title('Pipeline Hydraulics: Non-Newtonian vs. Newtonian Flow')
ax.grid(True, which='both', ls='--', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Sensitivity of Hydraulic Pressure Drop to Rheological Uncertainty
Evaluating how errors in estimated $K$ and $n$ propagate into hydraulic sizing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sensitivity to K
K_sweep = np.linspace(1.0, 5.0, 25)
dP_K = sensitivity_analysis(base_inp, 'K', K_sweep)
axes[0].plot(K_sweep, dP_K / 1000, 'r-o', lw=2, ms=5)
axes[0].set_xlabel('Consistency Index K (Pa·sⁿ)')
axes[0].set_ylabel('Pressure Drop ΔP (kPa)')
axes[0].set_title('Linear Sensitivity to Consistency Index (ΔP ∝ K)')
axes[0].grid(True, ls='--', alpha=0.3)

# Sensitivity to n
n_sweep = np.linspace(0.2, 0.9, 25)
dP_n = sensitivity_analysis(base_inp, 'n', n_sweep)
axes[1].plot(n_sweep, dP_n / 1000, 'g-s', lw=2, ms=5)
axes[1].set_xlabel('Flow Behavior Index n')
axes[1].set_ylabel('Pressure Drop ΔP (kPa)')
axes[1].set_title('Nonlinear Sensitivity to Shear-Thinning Exponent n')
axes[1].grid(True, ls='--', alpha=0.3)

plt.tight_layout()
plt.show()

### Engineering Conclusions
1. **Rabinowitsch Shear Rate Multiplication**: For pseudoplastic fluids ($n < 1$), wall shear rate $\dot{\gamma}_w$ is significantly elevated by factor $(3n+1)/(4n)$ compared to parabolic Newtonian flow ($8V/D$).
2. **Pumping Energy Advantage**: Because viscosity drops precipitously at high wall shear rates, the required pumping pressure drop $\Delta P$ scales as $V^n$ rather than $V^1$, providing immense thermodynamic energy savings at elevated flow velocities.
3. **Parameter Sensitivity**: While $\Delta P$ is strictly linear in $K$, it depends exponentially on $n$. Accurate estimation of $n$ via physics-guided ML is therefore the primary factor in preventing pipeline pump under-sizing.